In [1]:
import pandas as pd
import numpy as  np

In [2]:
# Cargar nuestro datos postproccesados.
X_EXPANDED_TEST = pd.read_pickle("data/postproccesed/X_EXPANDED.pkl")
y_train = pd.read_pickle("data/postproccesed/y.pkl")

In [3]:
X_EXPANDED_TEST.shape

(20974, 6835)

In [4]:
from sklearn.feature_selection import VarianceThreshold

lowVarianceFilter = VarianceThreshold(0.05)

In [5]:
lowVarianceFilter.fit(X_EXPANDED_TEST)

VarianceThreshold(threshold=0.05)

In [6]:
X_wo_low_variance = lowVarianceFilter.transform(X_EXPANDED_TEST)

In [7]:
X_wo_low_variance.shape

(20974, 2569)

In [8]:
X_wo_low_variancelow_df = pd.DataFrame(data=X_wo_low_variance,
                                        columns=lowVarianceFilter.get_feature_names_out(),
                                        index=X_EXPANDED_TEST.index)

In [9]:
X_wo_low_variancelow_df_2 = X_wo_low_variancelow_df.loc[:,~X_wo_low_variancelow_df.columns.duplicated()].copy()

In [10]:
X_wo_low_variancelow_df_2.shape

(20974, 2471)

In [11]:
from feature_engine.selection import DropCorrelatedFeatures, DropDuplicateFeatures


In [12]:
filter_duplicates = DropDuplicateFeatures()
filter_duplicates.fit(X_wo_low_variancelow_df_2)
X_wo_duplicates = filter_duplicates.transform(X_wo_low_variancelow_df_2)

In [13]:
X_wo_duplicates.shape

(20974, 1526)

In [14]:
correlated_filter = DropCorrelatedFeatures(threshold=0.9)

In [15]:
correlated_filter.fit(X_wo_duplicates)
X_wo_high_correlations = correlated_filter.transform(X_wo_duplicates)

In [16]:
X_wo_high_correlations.shape

(20974, 530)

In [17]:
from  feature_engine.selection import ProbeFeatureSelection
from sklearn.linear_model import LinearRegression

advanced_filtered = ProbeFeatureSelection(estimator=LinearRegression(),
                                          scoring='neg_mean_absolute_percentage_error',
                                          n_probes=3,
                                          distribution="normal",)

In [19]:
advanced_filtered.fit(X_wo_high_correlations, y_train)
final_X = advanced_filtered.transform(X_wo_high_correlations)

In [20]:
final_X.shape

(20974, 527)

In [21]:
final_X.to_pickle("data/filtered/X_filter_1.pkl")

In [22]:
# Vamos nuestro primer modelo.

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor()
rf_model.fit(final_X,y_train)

RandomForestRegressor()

In [23]:
rf_model.score(final_X,y_train)

0.8302118474915707

In [24]:
rf_model.predict(final_X.iloc[:2])

array([ 0.87802475, -0.0790222 ])

In [25]:
import pickle
with open('data/postproccesed/preproccesing_price_Transformer.pkl', 'rb') as f:
    price_transformer = pickle.load(f)

In [26]:
with open('data/postproccesed/preproccesing_scaler_price.pkl', 'rb') as f:
    scalar_price = pickle.load(f)

In [27]:

def unscale(model, data_points):
    prediction = model.predict(data_points)
    unscaled = scalar_price.inverse_transform(prediction.reshape(-1, 1))
    unnormed = price_transformer.inverse_transform(unscaled.reshape(-1, 1))
    return unnormed

In [28]:
unscale(rf_model, final_X.iloc[:2])

/Users/mmartin/opt/anaconda3/envs/modelizacion_datos/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(


array([[12000000.      ],
       [ 6499178.617316]])

In [29]:
# cogemos el set de entrenamiento original
test_data_df = pd.read_csv("data/preprocessed/test_data.csv")

In [30]:
test_data_df.shape

(8989, 41)

In [31]:
test_data_df = test_data_df.replace({9:np.nan})
test_data_df[test_data_df.columns[5:]] = test_data_df[test_data_df.columns[5:]].replace({0:'NO', 1:'SI', np.nan:'NO_DISPONIBLE'})

In [32]:
test_data_df.head()

,Price,city,Area,Location,No. of Bedrooms,Resale,MaintenanceStaff,Gymnasium,SwimmingPool,LandscapedGardens,...,LiftAvailable,BED,VaastuCompliant,Microwave,GolfCourse,TV,DiningTable,Sofa,Wardrobe,Stadium
0,2872000,Kolkata,883,Narendrapur,2.0,SI,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,...,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE
1,8400000,Bangalore,1400,Uttarahalli Main Road,3.0,NO,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,...,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE
2,2300000,Kolkata,1050,Garia,3.0,SI,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,...,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE
3,13000000,Delhi,1200,Azad Apartments,3.0,SI,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,...,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE
4,9324000,Bangalore,1335,Banashankari,2.0,NO,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,...,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE,NO_DISPONIBLE


In [33]:
with open('data/postproccesed/preproccesing_beds_imputer.pkl', 'rb') as f:
    beds_imputer = pickle.load(f)

In [34]:
test_data_df['No. of Bedrooms'] = beds_imputer.fit_transform(test_data_df[['No. of Bedrooms']])

In [35]:
with open('data/postproccesed/preproccesing_ohEncoder.pkl', 'rb') as f:
    oh_encoder = pickle.load(f)

In [36]:
cat_columns = ["city"] + test_data_df.columns[5:].to_list()
cat_columns
cat_oh_data = oh_encoder.transform(test_data_df[cat_columns])

df_cat_oh_data = pd.DataFrame(data=cat_oh_data, columns=oh_encoder.get_feature_names_out(), index=test_data_df.index)

In [37]:
with open('data/postproccesed/preproccesing_gapEncoder.pkl', 'rb') as f:
    gap_encoder = pickle.load(f)

In [38]:
cat_location_data = gap_encoder.transform(test_data_df['Location'])

In [39]:
with open('data/postproccesed/preproccesing_area_Transformer.pkl', 'rb') as f:
    area_transformer = pickle.load(f)

In [43]:
with open('data/postproccesed/preproccesing_beds_Transformer.pkl', 'rb') as f:
    bed_transformer = pickle.load(f)

In [44]:
with open('data/postproccesed/preproccesing_price_Transformer.pkl', 'rb') as f:
    price_transformer = pickle.load(f)

In [42]:
with open('data/postproccesed/preproccesing_gapEncoder.pkl', 'rb') as f:
    gap_encoder = pickle.load(f)